In [ ]:
! git clone https://github.com/juanmafdez/LLP_APP_Colombia

Cloning into 'LLP_APP_Colombia'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (274/274), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 274 (delta 104), reused 117 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (274/274), 2.25 MiB | 7.27 MiB/s, done.
Resolving deltas: 100% (104/104), done.


In [ ]:
from sklearn.model_selection import train_test_split
import os, sys, math, json, shutil, glob, random, time, datetime
from shapely.geometry import box
from scipy.stats import pearsonr
import geopandas as gpd
from pathlib import Path
import tensorflow as tf
from PIL import Image
import pandas as pd
import numpy as np
import tifffile

In [ ]:
sys.path.append('LLP_APP_Colombia/swav')
import architecture

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_path = '/content/drive/MyDrive/llp_col_project/datasets/sentinel'
final_output = "/content/train_data"
os.makedirs(final_output, exist_ok=True)
departments = {
    'tolima': 'tol_partitions_aschips_2c14c05f234fb'
}
for alias, folder_name in departments.items():
    print(f"--- Unzipping {alias} ---")
    !unzip -o -q {data_path}/{folder_name}.zip -d /content/
    full_path = f"/content/*/{folder_name}/sentinel2-rgb-median-2020"
    found_dirs = glob.glob(full_path)
    if found_dirs:
        src_dir = found_dirs[0]
        files = os.listdir(src_dir)
        print(f"Found files: {len(files)}")
        for f in files:
            src_file = os.path.join(src_dir, f)
            dst_file = os.path.join(final_output, f"{f}")
            shutil.move(src_file, dst_file)
        print(f"{alias} moved correctly.")
    else:
        print(f"{alias} not found")
    !rm -rf /content/gjson_data*
print(f"\n--- Process Finished ---")
print(f"Total files in {final_output}: {len(os.listdir(final_output))}")

--- Unzipping tolima ---
Found files: 23902
tolima moved correctly.

--- Process Finished ---
Total files in /content/train_data: 23902


In [ ]:
SRC_DIR = '/content/train_data' # .tif path
PNG_DIR = '/content/train_data_png8' # .png path
os.makedirs(PNG_DIR, exist_ok=True)
def save_png8_from_array(img_np: np.ndarray, out_path: str):
    if img_np.ndim == 3 and img_np.shape[0] < img_np.shape[1]:
        img_np = np.transpose(img_np, (1, 2, 0))
    elif img_np.ndim == 2:
        img_np = img_np[..., None]
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)
    elif img_np.shape[-1] > 3:
        img_np = img_np[..., :3]
    arr = img_np.astype(np.uint8)
    im = Image.fromarray(arr)  # RGB
    im.save(out_path, format='PNG', compress_level=0)  # PNG lossless
tif_paths = sorted(glob.glob(os.path.join(SRC_DIR, '*.tif')))
print(f"Converting {len(tif_paths)} TIFF a PNG 8-bit (lossless)...")
converted = 0
for p in tif_paths:
    out_path = os.path.join(PNG_DIR, Path(p).stem + '.png')
    if not os.path.exists(out_path):
        arr = tifffile.imread(p)
        save_png8_from_array(arr, out_path)
        converted += 1
total_png = len(glob.glob(os.path.join(PNG_DIR, '*.png')))
print(f"Done. Converted: {converted}. Total PNG created: {total_png}")

Converting 23902 TIFF a PNG 8-bit (lossless)...
Done. Converted: 23902. Total PNG created: 23902


In [ ]:
def load_png_uint8(path):
    im = Image.open(path)
    arr = np.array(im)  # uint8
    if arr.ndim == 2:
        arr = np.stack([arr]*3, axis=-1)
    elif arr.shape[-1] > 3:
        arr = arr[..., :3]
    return arr
samples = random.sample(tif_paths, min(10, len(tif_paths)))
mismatches = 0
for p in samples:
    t = tifffile.imread(p)
    if t.ndim == 3 and t.shape[0] < t.shape[1]:
        t = np.transpose(t, (1,2,0))
    if t.ndim == 2:
        t = np.stack([t]*3, axis=-1)
    elif t.shape[-1] > 3:
        t = t[..., :3]
    t = t.astype(np.uint8)
    png_path = os.path.join(PNG_DIR, Path(p).stem + '.png')
    p_arr = load_png_uint8(png_path)
    if not np.array_equal(t, p_arr):
        mismatches += 1
        print(f"Detected difference in: {p}")

if mismatches == 0:
    print("Verification: TIFF uint8 and PNG 8-bit are equivalents.")
else:
    print(f"Verification: {mismatches} differences in {len(samples)} samples.")

Verification: TIFF uint8 and PNG 8-bit are equivalents.


In [ ]:
def list_png_files(root):
    files = tf.io.gfile.glob(os.path.join(root, '*.png'))
    return sorted(files)
all_png = list_png_files(PNG_DIR)
print(f"PNG found: {len(all_png)}")

split_idx = int(0.8 * len(all_png))
train_files = all_png[:split_idx]
val_files   = all_png[split_idx:]

def decode_png(path):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_png(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([None, None, 3])
    return img

PNG found: 23902


In [ ]:
GJSON_CHIPS = '/content/drive/MyDrive/llp_col_project/datasets/sentinel/geojson_sentinel/tol_partitions_aschips_2c14c05f234fb.geojson'
GJSON_LABEL = '/content/drive/MyDrive/llp_col_project/datasets/esawc/tolima/tol_partitions_aschips_2c14c05f234fb.geojson'
IMG_ROOT    = '/content/train_data_png8'
BACKBONE_WEIGHTS = '/content/drive/MyDrive/llp_col_project/swav_checkpoints/feature_backbone_latest.weights.h5'

In [ ]:
def extraer_prop_ewc(d):
    if isinstance(d, str):
        try:
            d = json.loads(d)
        except Exception:
            import ast
            d = ast.literal_eval(d)
    return float(d.get("1", 0.0))
gdf_chips = gpd.read_file(GJSON_CHIPS)
gdf_lab   = gpd.read_file(GJSON_LABEL)
gdf_lab['prop_chip'] = gdf_lab['esaworldcover-2020_proportions'].apply(extraer_prop_ewc)
chips = gdf_chips.merge(gdf_lab[['identifier','prop_chip']], on='identifier', how='inner')
if chips.crs is None or not chips.crs.is_projected:
    chips = chips.to_crs(3116)

In [ ]:
def build_bags_municipios(
    chips_gdf,
    bag_col='foreignid_municipalities',
    split_col='split_municipalities'
):
    required = {'identifier', bag_col, 'prop_chip', 'geometry', split_col}
    missing = required - set(chips_gdf.columns)
    if missing:
        raise ValueError(f"Missing required columns in chips_gdf: {missing}")
    df = chips_gdf[['identifier', bag_col, 'prop_chip', 'geometry', split_col]].copy()
    df = df.rename(columns={bag_col: 'bag_id'})
    df['bag_id'] = df['bag_id'].astype('str')
    df = df[df[split_col].isin(['train', 'test'])].copy()
    if df.empty:
        raise ValueError("No chips with split_municipalities in {'train','test'}")
    per_bag_counts = df.groupby('bag_id')[split_col].nunique()
    if (per_bag_counts > 1).any():
        bad_bags = per_bag_counts[per_bag_counts > 1].index.tolist()
        raise RuntimeError(
            f"Found municipalities with mixed split train/test: {bad_bags[:10]} (sample). "
            f"Check GJSON_LABEL to ensure single split per municipality."
        )
    bag_targets = df.groupby('bag_id', as_index=False)['prop_chip'] \
                    .mean().rename(columns={'prop_chip':'pi'})
    bag_idx, uniques = pd.factorize(bag_targets['bag_id'], sort=True)
    bag_targets['bag_idx'] = bag_idx.astype('int64')
    map_id_to_idx = dict(zip(bag_targets['bag_id'], bag_targets['bag_idx']))
    df['bag_idx'] = df['bag_id'].map(map_id_to_idx).astype('int64')
    per_muni_split = df.groupby('bag_id')[split_col].first().to_dict()
    df['split_bag'] = df['bag_id'].map(per_muni_split).astype(str)
    cols = ['identifier','prop_chip','bag_id','bag_idx','split_bag','geometry']
    return df[cols], bag_targets[['bag_id','bag_idx','pi']]

In [ ]:
def build_grid(chips_gdf, area_km2=2.0, offset_frac=0.0, assign_mode='centroid'):
    import math
    import numpy as np
    import geopandas as gpd
    from shapely.geometry import box
    gdf = chips_gdf.copy()
    if not isinstance(gdf, gpd.GeoDataFrame):
        gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=None)
    if gdf.geometry.name != 'geometry':
        gdf = gdf.set_geometry('geometry', inplace=False)
    if gdf.crs is None or not gdf.crs.is_projected:
        gdf = gdf.to_crs(3116)
    side = math.sqrt(area_km2 * 1_000_000.0)
    minx, miny, maxx, maxy = gdf.total_bounds
    ox = offset_frac * side
    oy = offset_frac * side
    xs = np.arange(minx + ox, maxx + side, side)
    ys = np.arange(miny + oy, maxy + side, side)
    cells, ids = [], []
    k = 0
    for x in xs:
        for y in ys:
            cells.append(box(x, y, x + side, y + side))
            ids.append(k)
            k += 1
    grid = gpd.GeoDataFrame({'bag_id': ids, 'geometry': cells}, crs=gdf.crs)
    if grid.geometry.name != 'geometry':
        grid = grid.set_geometry('geometry', inplace=False)
    if assign_mode == 'centroid':
        pts = gdf[['identifier', 'prop_chip']].copy()
        pts['geometry'] = gdf.geometry.centroid
        pts = gpd.GeoDataFrame(pts, geometry='geometry', crs=gdf.crs)
        joined_pts = gpd.sjoin(
            pts, grid[['bag_id','geometry']], how='left', predicate='within'
        )
        missing_mask = joined_pts['bag_id'].isna()
        if missing_mask.any():
            pts_missing = joined_pts.loc[missing_mask, ['identifier','prop_chip','geometry']]
            joined_near = gpd.sjoin_nearest(
                pts_missing, grid[['bag_id','geometry']], how='left'
            )
            joined_pts.loc[missing_mask, 'bag_id'] = joined_near['bag_id'].values
        if joined_pts['bag_id'].isna().any():
            n_drop = int(joined_pts['bag_id'].isna().sum())
            print(f"[WARN] {n_drop} centroides no entraron en ninguna celda; se eliminarán.")
            joined_pts = joined_pts.dropna(subset=['bag_id'])
        joined = joined_pts[['identifier','prop_chip','bag_id']].merge(
            gdf[['identifier','geometry']], on='identifier', how='left'
        )
        joined = gpd.GeoDataFrame(joined, geometry='geometry', crs=gdf.crs)
    else:
        joined = gpd.sjoin(
            gdf[['identifier','prop_chip','geometry']], grid[['bag_id','geometry']],
            how='inner', predicate='intersects'
        )
        joined = joined[['identifier','prop_chip','bag_id','geometry']]
        joined = gpd.GeoDataFrame(joined, geometry='geometry', crs=gdf.crs)
    if joined['identifier'].duplicated(keep=False).any():
        before = len(joined)
        joined = joined.drop_duplicates(subset=['identifier'], keep='first')
        after = len(joined)
        print(f"[INFO] Eliminadas {before - after} filas duplicadas por identifier.")
    joined['bag_id'] = joined['bag_id'].astype('int64')
    bag_targets = joined.groupby('bag_id', as_index=False)['prop_chip'] \
                        .mean().rename(columns={'prop_chip':'pi'})
    bag_targets['bag_id'] = bag_targets['bag_id'].astype('int64')
    bag_idx, uniques = pd.factorize(bag_targets['bag_id'], sort=True)
    bag_targets['bag_idx'] = bag_idx.astype('int64')
    bag_map = dict(zip(bag_targets['bag_id'].values, bag_targets['bag_idx'].values))
    joined['bag_idx'] = joined['bag_id'].map(bag_map).astype('int64')
    joined = joined[['identifier','prop_chip','bag_id','bag_idx','geometry']]
    return joined, bag_targets[['bag_id','bag_idx','pi']]

In [ ]:
SEED=666
def split_bags(bag_targets, train_frac=0.75, seed=SEED):
    bag_ids = bag_targets['bag_id'].values
    bag_train, bag_test = train_test_split(bag_ids, test_size=1-train_frac, random_state=seed)
    return set(bag_train), set(bag_test)

def tag_split_by_bag(df, bag_train_set):
    df = df.copy()
    df['split_bag'] = np.where(df['bag_id'].isin(bag_train_set),'train','test')
    return df

In [ ]:
def list_image_paths(root_dir, identifiers, ext='.png'):
    paths = []
    for ident in identifiers:
        p = os.path.join(root_dir, f"{ident}{ext}")
        if os.path.exists(p):
            paths.append(p)
        else:
            p2 = os.path.join(root_dir, f"{ident}.tif")
            if os.path.exists(p2):
                paths.append(p2)
    return paths

def make_tf_dataset(df_split, img_root, bag_targets, batch_size=128, shuffle=True, seed=SEED):
    B = int(bag_targets['bag_idx'].max()) + 1
    y_true = np.zeros((B,), dtype=np.float32)
    y_true[bag_targets['bag_idx'].values] = bag_targets['pi'].values.astype(np.float32)
    y_true_bag = tf.constant(y_true, dtype=tf.float32)
    bag_groups = {}
    for row in df_split[['identifier','bag_idx']].itertuples(index=False):
        ident = row.identifier
        bidx  = int(row.bag_idx)
        p = os.path.join(img_root, f"{ident}.png")
        if not os.path.exists(p):
            raise FileNotFoundError(f"PNG no encontrado para identifier={ident}")
        bag_groups.setdefault(bidx, []).append(p)
    bag_idx_list = list(bag_groups.keys())
    rng = np.random.default_rng(seed)
    if shuffle:
        rng.shuffle(bag_idx_list)

    def gen_paths_and_labels():
        for b in bag_idx_list:
            paths = bag_groups[b]
            if shuffle:
                rng.shuffle(paths)
            for p in paths:
                yield p, b

    ds_paths = tf.data.Dataset.from_generator(
        gen_paths_and_labels,
        output_signature=(
            tf.TensorSpec(shape=(), dtype=tf.string),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )

    def _decode_png(path, bidx):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_png(img_bytes, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        return img, bidx

    ds = ds_paths.map(_decode_png, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, y_true_bag

In [ ]:
def make_model(backbone_weights_path, train_backbone=False):
    backbone = architecture.get_resnet_backbone()
    if isinstance(backbone_weights_path, str):
        if backbone_weights_path.strip() and os.path.exists(backbone_weights_path):
            backbone.load_weights(backbone_weights_path)
        else:
            pass
    else:
        pass
    for l in backbone.layers:
        l.trainable = train_backbone
    x_in = tf.keras.Input(shape=(None,None,3), dtype=tf.float32)
    feat = backbone(x_in, training=False)
    out = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')(feat)
    model = tf.keras.Model(x_in, out)
    return model

In [ ]:
@tf.function
def llp_bag_mse(preds_chip, bag_idx, y_true_bag, weighted=True):
    bag_idx = tf.cast(bag_idx, tf.int32)
    preds_chip = tf.cast(tf.squeeze(preds_chip, axis=-1), tf.float32)
    uniq, idx = tf.unique(bag_idx)
    num_uniq = tf.shape(uniq)[0]
    sums   = tf.math.unsorted_segment_sum(preds_chip, idx, num_uniq)
    counts = tf.math.unsorted_segment_sum(tf.ones_like(preds_chip), idx, num_uniq)
    mean_pred = sums / (counts + 1e-8)
    y_true_sel = tf.gather(y_true_bag, uniq)
    per_bag_sq = tf.square(mean_pred - y_true_sel)
    if weighted:
        w = counts / (tf.reduce_sum(counts) + 1e-8)
        return tf.reduce_sum(w * per_bag_sq)
    else:
        return tf.reduce_mean(per_bag_sq)

@tf.function(reduce_retracing=True)
def train_step_llp(model, optimizer, images, bag_idx, y_true_bag):
    with tf.GradientTape() as tape:
        preds = model(images, training=True)
        loss  = llp_bag_mse(preds, bag_idx, y_true_bag)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [ ]:
def train_llp(model, train_ds, y_true_bag, val_ds=None, y_true_bag_val=None,
              epochs=10, lr=1e-4):
    warm_images = None
    warm_bags   = None
    for _imgs, _bags in train_ds.take(1):
        warm_images = _imgs
        warm_bags   = _bags
    if warm_images is None:
        raise RuntimeError("train_ds is empty; cannot initialize the train.")
    _ = model(warm_images, training=False)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    opt.build(model.trainable_variables)
    hist = []
    for ep in range(1, epochs+1):
        losses = []
        for batch_imgs, batch_bag_idx in train_ds:
            loss = train_step_llp(model, opt, batch_imgs, batch_bag_idx, y_true_bag)
            losses.append(float(loss.numpy()))
        ep_loss = float(np.mean(losses))
        log = f"[Epoch {ep:03d}] train_bagMSE={ep_loss:.4f}"
        if val_ds is not None and y_true_bag_val is not None:
            y_pred_bag, y_true_b = aggregate_bag_preds(model, val_ds, y_true_bag_val)
            rmse, r = rmse_corr(y_true_b, y_pred_bag)
            log += f" | val_RMSE={rmse:.4f} r={r:.3f}"
        print(log)
        hist.append(ep_loss)
    return hist

def aggregate_bag_preds(model, ds, y_true_bag):
    sums = None
    counts = None
    B = int(y_true_bag.shape[0])
    sums   = np.zeros((B,), dtype=np.float64)
    counts = np.zeros((B,), dtype=np.int64)
    for batch_imgs, batch_bag_idx in ds:
        preds = model.predict(batch_imgs, verbose=0).squeeze(-1)
        bi = batch_bag_idx.numpy()
        for p, b in zip(preds, bi):
            sums[b] += float(p)
            counts[b] += 1
    mask = counts > 0
    y_pred_bag = np.zeros_like(sums, dtype=np.float32)
    y_pred_bag[mask] = (sums[mask] / counts[mask]).astype(np.float32)
    y_true = y_true_bag.numpy().astype(np.float32)
    return y_pred_bag[mask], y_true[mask]

def rmse_corr(y_true, y_pred):
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    r, _ = pearsonr(y_true, y_pred) if len(y_true) > 2 else (np.nan, np.nan)
    return rmse, float(r)

In [ ]:
def debug_check_grid_assignment(df):
    dup_counts = df['identifier'].value_counts()
    n_dup = int((dup_counts > 1).sum())
    print(f"[Debug] Mapped chips to >1 bag: {n_dup} (of {len(dup_counts)} chips)")
    if n_dup > 0:
        print("[Debug] Ex duplicated chips:", dup_counts[dup_counts > 1].head(10).to_dict())

In [ ]:
def run_experiment_grids(chips_gdf, img_root, backbone_weights, areas_km2, offsets=[0.0],
                         epochs=10, lr=1e-4, train_backbone=False):
    results = []
    for a in areas_km2:
        for off in offsets:
            print(f"\n=== Grid {a} km² | offset={off} ===")
            df, bag_targets = build_grid(chips_gdf, area_km2=a, offset_frac=off)
            debug_check_grid_assignment(df)
            train_bags, test_bags = split_bags(bag_targets, train_frac=0.75, seed=SEED)
            df = df.copy()
            df['split_bag'] = np.where(df['bag_id'].isin(train_bags),'train','test')
            train_df = df[df['split_bag']=='train']
            test_df  = df[df['split_bag']=='test']
            train_ds, y_true_bag      = make_tf_dataset(train_df, img_root, bag_targets, batch_size=128, shuffle=True)
            test_ds,  y_true_bag_test = make_tf_dataset(test_df,  img_root, bag_targets, batch_size=128, shuffle=False)
            model = make_model(backbone_weights, train_backbone=train_backbone)
            train_llp(model, train_ds, y_true_bag, val_ds=test_ds, y_true_bag_val=y_true_bag_test, epochs=epochs, lr=lr)
            y_pred_bag, y_true_b = aggregate_bag_preds(model, test_ds, y_true_bag_test)
            rmse, r = rmse_corr(y_true_b, y_pred_bag)
            print(f"[Grid {a} km²] TEST bag-level: RMSE={rmse:.4f} r={r:.3f}")
            results.append({'area_km2':a, 'offset':off, 'rmse':rmse, 'r':r})
    return results

In [ ]:
OUT_DIR = '/content/drive/MyDrive/llp_col_project/experimental_results'
os.makedirs(OUT_DIR, exist_ok=True)
RAW_CSV      = os.path.join(OUT_DIR, 'results_raw_speed.csv')
SUMMARY_CSV  = os.path.join(OUT_DIR, 'results_summary_speed.csv')
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
def _append_row_csv(csv_path: str, row_dict: dict):
    df_row = pd.DataFrame([row_dict])
    if not os.path.exists(csv_path):
        df_row.to_csv(csv_path, index=False)
    else:
        df_row.to_csv(csv_path, mode='a', header=False, index=False)
def _recompute_summary(raw_csv_path: str, summary_csv_path: str):
    if not os.path.exists(raw_csv_path):
        return
    df = pd.read_csv(raw_csv_path)
    if 'stage' in df.columns:
        dff = df[df['stage']=='final'].copy()
    else:
        dff = df.copy()
    if dff.empty:
        return
    grp = dff.groupby(['dataset_tag','method_tag','area_km2','offset'], as_index=False)
    agg = grp.agg(
        B=('num_bags','mean'),
        r_mean=('r','mean'),
        r_std =('r','std'),
        rmse_mean=('rmse','mean'),
        rmse_std =('rmse','std'),
        seeds_completed=('seed','nunique'),
        runs=('stage','count')
    )
    agg = agg.sort_values(['dataset_tag','method_tag','area_km2','offset'])
    agg.to_csv(summary_csv_path, index=False)

def _evaluate(model, ds, y_true_bag):
    """Agrega predicciones por bolsa y calcula RMSE, r sobre bolsas presentes en ds."""
    y_pred_bag, y_true_b = aggregate_bag_preds(model, ds, y_true_bag)
    rmse, r = rmse_corr(y_true_b, y_pred_bag)
    return float(rmse), float(r), int(len(y_pred_bag))

def fit_llp_with_checkpoints(model,
                             train_ds, y_true_bag_train,
                             val_ds,   y_true_bag_val,
                             optimizer, total_epochs=30, save_every=10,
                             partial_save_fn=None):
    optimizer.build(model.trainable_variables)
    hist = []
    epochs_done = 0
    while epochs_done < total_epochs:
        block = min(save_every, total_epochs - epochs_done)
        for ep in range(block):
            losses = []
            for batch_imgs, batch_bidx in train_ds:
                loss = train_step_llp(model, optimizer, batch_imgs, batch_bidx, y_true_bag_train)
                losses.append(float(loss.numpy()))
            hist.append(float(np.mean(losses)))
        epochs_done += block
        if partial_save_fn is not None:
            rmse_val, r_val, B_val = (np.nan, np.nan, 0)
            if (val_ds is not None) and (y_true_bag_val is not None):
                rmse_val, r_val, B_val = _evaluate(model, val_ds, y_true_bag_val)
            partial_save_fn(epochs_done, hist[-1], rmse_val, r_val, B_val)
    return hist

def launch_llp_experiments(chips_gdf,
                           img_root,
                           backbone_weights,
                           train_backbone,
                           areas_km2,
                           offsets,
                           seeds,
                           dataset_tag="Tolima",
                           method_tag=None,
                           epochs=10,
                           lr=1e-4,
                           batch_size=128,
                           save_every=10):
    if method_tag is None:
        method_tag = "pretrained" if (isinstance(backbone_weights, str) and os.path.exists(backbone_weights)) else "scratch"
    os.makedirs(OUT_DIR, exist_ok=True)
    for area in areas_km2:
        for off in offsets:
            for seed in seeds:
                set_global_seed(int(seed))
                df, bag_targets = build_grid(chips_gdf, area_km2=float(area), offset_frac=float(off), assign_mode='centroid')
                train_bags, test_bags = split_bags(bag_targets, train_frac=0.75, seed=int(seed))
                df2 = df.copy()
                df2['split_bag'] = np.where(df2['bag_id'].isin(train_bags),'train','test')
                train_df = df2[df2['split_bag']=='train']
                test_df  = df2[df2['split_bag']=='test']
                train_ds, y_true_bag_tr = make_tf_dataset(train_df, img_root, bag_targets, batch_size=batch_size, shuffle=True)
                val_ds,   y_true_bag_va = make_tf_dataset(test_df,  img_root, bag_targets, batch_size=batch_size, shuffle=False)
                model = make_model(backbone_weights_path=backbone_weights, train_backbone=train_backbone)
                for _imgs, _b in train_ds.take(1):
                    _ = model(_imgs, training=train_backbone)
                opt = tf.keras.optimizers.Adam(learning_rate=lr)
                t0 = time.time()
                def _on_partial_save(epochs_done, train_loss_last, rmse_val, r_val, B_val):
                    row = {
                        'timestamp'   : time.strftime('%Y-%m-%d %H:%M:%S'),
                        'dataset_tag' : dataset_tag,
                        'method_tag'  : method_tag,
                        'area_km2'    : float(area),
                        'offset'      : float(off),
                        'seed'        : int(seed),
                        'epoch'       : int(epochs_done),
                        'train_bagMSE': float(train_loss_last),
                        'rmse'        : float(rmse_val) if not np.isnan(rmse_val) else '',
                        'r'           : float(r_val)    if not np.isnan(r_val)    else '',
                        'num_bags'    : int(B_val),
                        'stage'       : 'interim',
                        'elapsed_s'   : float(time.time()-t0),
                    }
                    _append_row_csv(RAW_CSV, row)
                    _recompute_summary(RAW_CSV, SUMMARY_CSV)
                hist = fit_llp_with_checkpoints(
                    model,
                    train_ds, y_true_bag_tr,
                    val_ds,   y_true_bag_va,
                    optimizer=opt,
                    total_epochs=int(epochs),
                    save_every=int(save_every),
                    partial_save_fn=_on_partial_save
                )
                rmse_test, r_test, B_test = _evaluate(model, val_ds, y_true_bag_va)
                row_final = {
                    'timestamp'   : time.strftime('%Y-%m-%d %H:%M:%S'),
                    'dataset_tag' : dataset_tag,
                    'method_tag'  : method_tag,
                    'area_km2'    : float(area),
                    'offset'      : float(off),
                    'seed'        : int(seed),
                    'epoch'       : int(epochs),
                    'train_bagMSE': float(hist[-1]) if len(hist)>0 else float('nan'),
                    'rmse'        : float(rmse_test),
                    'r'           : float(r_test),
                    'num_bags'    : int(B_test),
                    'stage'       : 'final',
                    'elapsed_s'   : float(time.time()-t0),
                }
                _append_row_csv(RAW_CSV, row_final)
                _recompute_summary(RAW_CSV, SUMMARY_CSV)
                print(f"[Done] dataset={dataset_tag} method={method_tag} area={area} offset={off} seed={seed} "
                      f"=> test RMSE={rmse_test:.4f} r={r_test:.3f}")


In [ ]:
AREAS   = [1, 32, 64, 1024]
OFFSETS = [0.0, 0.5]
SEEDS   = [666, 333]

launch_llp_experiments(
    chips_gdf=chips,
    img_root=IMG_ROOT,
    backbone_weights=None,
    train_backbone=True,
    areas_km2=AREAS,
    offsets=OFFSETS,
    seeds=SEEDS,
    dataset_tag="Tolima",
    method_tag="scratch",
    epochs=10,
    lr=1e-4,
    batch_size=128,
    save_every=1
)

[Done] dataset=Tolima method=scratch area=1 offset=0.0 seed=666 => test RMSE=0.1385 r=0.903
[Done] dataset=Tolima method=scratch area=1 offset=0.0 seed=333 => test RMSE=0.1751 r=0.896
[Done] dataset=Tolima method=scratch area=1 offset=0.5 seed=666 => test RMSE=0.2098 r=0.864
[Done] dataset=Tolima method=scratch area=1 offset=0.5 seed=333 => test RMSE=0.1229 r=0.916
[Done] dataset=Tolima method=scratch area=32 offset=0.0 seed=666 => test RMSE=0.1182 r=0.923
[Done] dataset=Tolima method=scratch area=32 offset=0.0 seed=333 => test RMSE=0.1175 r=0.907
[Done] dataset=Tolima method=scratch area=32 offset=0.5 seed=666 => test RMSE=0.1072 r=0.920
[Done] dataset=Tolima method=scratch area=32 offset=0.5 seed=333 => test RMSE=0.0980 r=0.933
[Done] dataset=Tolima method=scratch area=64 offset=0.0 seed=666 => test RMSE=0.0968 r=0.933
[Done] dataset=Tolima method=scratch area=64 offset=0.0 seed=333 => test RMSE=0.1354 r=0.881
[Done] dataset=Tolima method=scratch area=64 offset=0.5 seed=666 => test R

In [ ]:
gdf_chips_m = gpd.read_file(GJSON_CHIPS)
gdf_lab_m   = gpd.read_file(GJSON_LABEL)

gdf_lab_m['prop_chip'] = gdf_lab_m['esaworldcover-2020_proportions'].apply(extraer_prop_ewc)
chips_m = gdf_chips.merge(gdf_lab_m[['identifier','prop_chip','foreignid_municipalities','split_municipalities']], on='identifier', how='inner')

In [ ]:
chips_m.head(1)

,area_km2,identifier,geometry,prop_chip,foreignid_municipalities,split_municipalities
0,0.998002,2a3a559fde032,"POLYGON ((-76.10181 3.06794, -76.10181 3.07698...",0.0,25f79caad246b,train


In [ ]:
MUNI_OUT_DIR = '/content/drive/MyDrive/llp_col_project/experimental_results'
MUNI_RAW_CSV = os.path.join(MUNI_OUT_DIR, 'results_raw_muni.csv')
os.makedirs(MUNI_OUT_DIR, exist_ok=True)

def _append_row_csv(csv_path: str, row_dict: dict):
    df_row = pd.DataFrame([row_dict])
    if not os.path.exists(csv_path):
        df_row.to_csv(csv_path, index=False)
    else:
        df_row.to_csv(csv_path, mode='a', header=False, index=False)

def _bag_upper_bound(df_split, bag_targets):
    pred_bag = df_split.groupby('bag_idx', as_index=False)['prop_chip'] \
                       .mean().rename(columns={'prop_chip':'p_hat'})
    gt_bag   = bag_targets[['bag_idx','pi']]
    m = pred_bag.merge(gt_bag, on='bag_idx', how='inner')
    y_true = m['pi'].values
    y_pred = m['p_hat'].values
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    if len(y_true) > 2:
        from scipy.stats import pearsonr
        r = float(pearsonr(y_true, y_pred)[0])
    else:
        r = np.nan
    return rmse, r, int(len(m))

def run_experiment_municipios(
    chips_gdf,
    img_root,
    backbone_weights,
    epochs=10,
    lr=1e-4,
    train_backbone=False,
    seed=111,
    dataset_tag="Tolima",
    method_tag=None,
    save_csv=True,
    verbose=True
):
    if method_tag is None:
        loaded_flag = (isinstance(backbone_weights, str) and bool(backbone_weights))
        method_tag = "swav_muni" if loaded_flag else "scratch_muni"
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    df_muni, bag_targets = build_bags_municipios(chips_gdf)
    train_df = df_muni[df_muni['split_bag']=='train'].copy()
    test_df  = df_muni[df_muni['split_bag']=='test'].copy()
    n_bags_train = train_df['bag_id'].nunique()
    n_bags_test  = test_df['bag_id'].nunique()
    n_chips_train = len(train_df)
    n_chips_test  = len(test_df)
    if verbose:
        print(f"[Municipal] #bags train={n_bags_train} | #bags test={n_bags_test} | "
              f"#chips train={n_chips_train} | #chips test={n_chips_test} | seed={seed}")
    train_ds, y_true_bag      = make_tf_dataset(train_df, img_root, bag_targets, batch_size=128, shuffle=True)
    test_ds,  y_true_bag_test = make_tf_dataset(test_df,  img_root, bag_targets, batch_size=128, shuffle=False)
    model = make_model(backbone_weights_path=backbone_weights, train_backbone=train_backbone)
    for _imgs, _b in train_ds.take(1):
        _ = model(_imgs, training=train_backbone)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    opt.build(model.trainable_variables)
    loaded = isinstance(backbone_weights, str) and bool(backbone_weights)
    if verbose:
        print(f"[Model] Pretrained loaded: {loaded} | train_backbone={train_backbone}")
    t0 = time.time()
    hist = train_llp(model, train_ds, y_true_bag, val_ds=test_ds, y_true_bag_val=y_true_bag_test, epochs=epochs, lr=lr)
    train_time_s = float(time.time() - t0)
    train_bagMSE_last = float(hist[-1]) if hist else float('nan')
    y_pred_bag, y_true_b = aggregate_bag_preds(model, test_ds, y_true_bag_test)
    rmse, r = rmse_corr(y_true_b, y_pred_bag)
    rmse = float(rmse); r = float(r)
    if verbose:
        print(f"[Municipal] TEST bag-level: RMSE={rmse:.4f} r={r:.3f}")
    rmse_oracle, r_oracle, B_oracle = _bag_upper_bound(test_df, bag_targets)
    if save_csv:
        row_final = {
            'timestamp'    : datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'dataset_tag'  : dataset_tag,
            'method_tag'   : method_tag,
            'bags_type'    : 'municipality',
            'area_km2'     : -1,
            'offset'       : '',
            'seed'         : int(seed),
            'epoch'        : int(epochs),
            'train_bagMSE' : train_bagMSE_last,
            'rmse'         : rmse,
            'r'            : r,
            'num_bags_train': int(n_bags_train),
            'num_bags_test' : int(n_bags_test),
            'num_chips_train': int(n_chips_train),
            'num_chips_test' : int(n_chips_test),
            # Oracle context (upper bound on TEST)
            'rmse_oracle'  : rmse_oracle,
            'r_oracle'     : r_oracle,
            'num_bags_test_oracle': int(B_oracle),
            # timing
            'elapsed_s'    : train_time_s,
            'stage'        : 'final'
        }
        _append_row_csv(MUNI_RAW_CSV, row_final)

    return model, (rmse, r)

In [ ]:
for seed in [333, 666]:
    _model, (rmse, r) = run_experiment_municipios(
        chips_gdf=chips_m,
        img_root=IMG_ROOT,
        backbone_weights=BACKBONE_WEIGHTS,
        epochs=10, lr=1e-4,
        train_backbone=True,
        seed=seed,
        dataset_tag="Tolima",
        method_tag="swav_muni",
        save_csv=True,
        verbose=True
    )

[Municipal] #bags train=25 | #bags test=14 | #chips train=13629 | #chips test=6485 | seed=333
[Model] Pretrained loaded: True | train_backbone=True
[Epoch 001] train_bagMSE=0.0443 | val_RMSE=0.1597 r=0.521
[Epoch 002] train_bagMSE=0.0188 | val_RMSE=0.1257 r=0.663
[Epoch 003] train_bagMSE=0.0111 | val_RMSE=0.0915 r=0.794
[Epoch 004] train_bagMSE=0.0102 | val_RMSE=0.1309 r=0.838
[Epoch 005] train_bagMSE=0.0093 | val_RMSE=0.1624 r=0.854
[Epoch 006] train_bagMSE=0.0101 | val_RMSE=0.1583 r=0.861
[Epoch 007] train_bagMSE=0.0091 | val_RMSE=0.1125 r=0.896
[Epoch 008] train_bagMSE=0.0085 | val_RMSE=0.0973 r=0.912
[Epoch 009] train_bagMSE=0.0081 | val_RMSE=0.0927 r=0.888
[Epoch 010] train_bagMSE=0.0087 | val_RMSE=0.1652 r=0.852
[Municipal] TEST bag-level: RMSE=0.1652 r=0.852
[Municipal] #bags train=25 | #bags test=14 | #chips train=13629 | #chips test=6485 | seed=666
[Model] Pretrained loaded: True | train_backbone=True
[Epoch 001] train_bagMSE=0.0265 | val_RMSE=0.1628 r=0.520
[Epoch 002] train_

In [ ]:
for seed in [333, 666]:
    _model, (rmse, r) = run_experiment_municipios(
        chips_gdf=chips_m,
        img_root=IMG_ROOT,
        backbone_weights=None,
        epochs=10, lr=1e-4,
        train_backbone=True,
        seed=seed,
        dataset_tag="Tolima",
        method_tag="scratch_muni",
        save_csv=True,
        verbose=True
    )

[Municipal] #bags train=25 | #bags test=14 | #chips train=13629 | #chips test=6485 | seed=333
[Model] Pretrained loaded: False | train_backbone=True
[Epoch 001] train_bagMSE=0.0180 | val_RMSE=0.1420 r=0.488
[Epoch 002] train_bagMSE=0.0107 | val_RMSE=0.3294 r=0.087
[Epoch 003] train_bagMSE=0.0181 | val_RMSE=0.3617 r=0.559
[Epoch 004] train_bagMSE=0.0099 | val_RMSE=0.4766 r=0.557
[Epoch 005] train_bagMSE=0.0070 | val_RMSE=0.4591 r=0.514
[Epoch 006] train_bagMSE=0.0071 | val_RMSE=0.2188 r=0.713
[Epoch 007] train_bagMSE=0.0060 | val_RMSE=0.2302 r=0.846
[Epoch 008] train_bagMSE=0.0064 | val_RMSE=0.1318 r=0.806
[Epoch 009] train_bagMSE=0.0066 | val_RMSE=0.1231 r=0.837
[Epoch 010] train_bagMSE=0.0064 | val_RMSE=0.1015 r=0.862
[Municipal] TEST bag-level: RMSE=0.1015 r=0.862
[Municipal] #bags train=25 | #bags test=14 | #chips train=13629 | #chips test=6485 | seed=666
[Model] Pretrained loaded: False | train_backbone=True
[Epoch 001] train_bagMSE=0.0233 | val_RMSE=0.2843 r=0.151
[Epoch 002] trai